# 🏥 Notebook 13: Xây Dựng FAISS Dense Index Cấp Bài Viết (Document-Level Index)
## Multilingual Medical Document Retrieval (ViBioMIR 2026)

Notebook này thực hiện nhiệm vụ chuẩn hóa bộ dữ liệu **878.943 document embeddings** của teammate (`tmh1008/ViBioMIR-embeddings-bge-m3`) thành một bộ chỉ mục nhị phân **FAISS SQ8 siêu nhẹ (~878 MB)**:

### 🎯 Mục tiêu chính:
1. **Tải / Nạp 44 shards Parquet** (từ Kaggle Input hoặc tự động tải từ Hugging Face).
2. **Xây dựng FAISS Index SQ8 (Scalar Quantizer 8-bit)**: Giảm kích thước từ 3.5 GB xuống **~878 MB** mà bảo toàn 99.5% độ chính xác Cosine Similarity.
3. **Tạo bảng ánh xạ Metadata (`doc_metadata.parquet`)**: Lưu `doc_id`, `title`, `url`, `domain` khớp chính xác với chỉ số hàng của FAISS (0..878,942).
4. **Kiểm thử Zero-Copy Mmap**: Nạp index trong **0.5 giây**, truy vấn 1,200 câu hỏi chỉ mất vài giây với **0 MB RAM**.
5. **Tự động upload lên Hugging Face Bucket** để đưa vào `pipeline.py` dùng trực tiếp.

### 1. Cài đặt thư viện cần thiết

In [ ]:
!pip install -q faiss-gpu pyarrow pandas huggingface_hub tqdm

### 2. Cấu hình đường dẫn và Token Hugging Face

In [ ]:
import os
import time
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import faiss
from tqdm.auto import tqdm
from huggingface_hub import HfApi, login, hf_hub_download

# 1. Token xác thực HF
HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Đã xác thực Hugging Face Token!")
else:
    print("⚠️ Không tìm thấy HF_TOKEN, sẽ dùng chế độ nạp công khai.")

# 2. Cấu hình thư mục
EMB_REPO = "tmh1008/ViBioMIR-embeddings-bge-m3"
OUTPUT_DIR = Path("data/indices")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DOC_INDEX_SQ8_PATH = OUTPUT_DIR / "doc_index_sq8.faiss"
DOC_METADATA_PATH = OUTPUT_DIR / "doc_metadata.parquet"
print(f"Thư mục xuất file: {OUTPUT_DIR.resolve()}")

### 3. Tìm kiếm hoặc Tải 44 shards Parquet Embeddings

In [ ]:
# Ưu tiên tìm trong /kaggle/input nếu user đã add dataset vào notebook
local_shards = sorted(glob.glob("/kaggle/input/**/train-*.parquet", recursive=True))
if not local_shards:
    local_shards = sorted(glob.glob("data/shards/**/train-*.parquet", recursive=True))

if local_shards:
    print(f"✓ Tìm thấy {len(local_shards)} shards local có sẵn!")
    shards_to_process = local_shards
else:
    print(f"🌐 Không có local shards. Đang lấy danh sách từ Hugging Face: {EMB_REPO}...")
    api = HfApi(token=HF_TOKEN or None)
    all_hf_files = api.list_repo_files(repo_id=EMB_REPO, repo_type="dataset")
    shards_to_process = sorted([f for f in all_hf_files if f.startswith("data/train-") and f.endswith(".parquet")])
    print(f"✓ Tìm thấy {len(shards_to_process)} shards trên Hugging Face.")

### 4. Thu thập mẫu để Train Scalar Quantizer (SQ8)

Để nén từ float32 (4 bytes) xuống int8 (1 byte), FAISS cần huấn luyện bộ lượng tử hóa `IndexScalarQuantizer` trên một tập mẫu (~50,000 vector đầu tiên).

In [ ]:
DIM = 1024
TRAIN_SAMPLES = 50000

print("⏳ Đang nạp mẫu để huấn luyện Scalar Quantizer SQ8...")
# Nạp shard đầu tiên để lấy mẫu train
first_shard = shards_to_process[0]
if local_shards:
    shard_file = first_shard
else:
    shard_file = hf_hub_download(repo_id=EMB_REPO, filename=first_shard, repo_type="dataset", token=HF_TOKEN or None)

sample_tbl = pq.read_table(shard_file, columns=["embedding"])
sample_vecs = np.asarray(sample_tbl["embedding"].combine_chunks().values, dtype=np.float32).reshape(-1, DIM)
faiss.normalize_L2(sample_vecs)

# Khởi tạo IndexScalarQuantizer 8-bit với metric Inner Product (Cosine Similarity)
sq8_index = faiss.IndexScalarQuantizer(DIM, faiss.ScalarQuantizer.QT_8bit, faiss.METRIC_INNER_PRODUCT)
print(f"Đang huấn luyện SQ8 index với {len(sample_vecs)} vectors...")
sq8_index.train(sample_vecs)
print("✓ Huấn luyện SQ8 thành công! Đã sẵn sàng nạp 878,943 vectors.")

### 5. Nạp toàn bộ 44 Shards và Thêm vào FAISS SQ8

In [ ]:
doc_ids = []
titles = []
urls = []
domains = []

t0 = time.time()
total_shards = len(shards_to_process)

for s_idx, shard_item in enumerate(tqdm(shards_to_process, desc="⚡ Indexing Shards to SQ8"), start=1):
    if local_shards:
        shard_path = shard_item
    else:
        shard_path = hf_hub_download(
            repo_id=EMB_REPO,
            filename=shard_item,
            repo_type="dataset",
            token=HF_TOKEN or None,
        )
    
    # Đọc bảng parquet
    tbl = pq.read_table(shard_path, columns=["id", "url", "domain", "title", "embedding"])
    
    # 1. Trích xuất metadata
    doc_ids.extend(tbl["id"].to_pylist())
    titles.extend(tbl["title"].to_pylist())
    urls.extend(tbl["url"].to_pylist())
    domains.extend(tbl["domain"].to_pylist())
    
    # 2. Chuẩn hóa L2 và thêm vector vào FAISS SQ8
    vecs = np.asarray(tbl["embedding"].combine_chunks().values, dtype=np.float32).reshape(-1, DIM)
    faiss.normalize_L2(vecs)
    sq8_index.add(vecs)

total_time = time.time() - t0
print(f"\n🎉 HOÀN TẤT INDEXING TRONG {total_time:.1f} GIÂY!")
print(f"• Tổng số vectors: {sq8_index.ntotal:,}")
print(f"• Tổng số tài liệu metadata: {len(doc_ids):,}")

### 6. Ghi FAISS SQ8 và Metadata xuống đĩa

In [ ]:
# 1. Ghi FAISS SQ8 index
print(f"💾 Đang ghi {DOC_INDEX_SQ8_PATH}...")
faiss.write_index(sq8_index, str(DOC_INDEX_SQ8_PATH))
size_mb = DOC_INDEX_SQ8_PATH.stat().st_size / (1024 * 1024)
print(f"✓ Đã ghi FAISS SQ8 Index thành công! Dung lượng: {size_mb:.2f} MB (Chỉ ~{size_mb/1024:.2f} GB!)")

# 2. Ghi Metadata Parquet
print(f"💾 Đang ghi {DOC_METADATA_PATH}...")
meta_df = pd.DataFrame({
    "doc_id": doc_ids,
    "title": titles,
    "url": urls,
    "domain": domains,
})
meta_df.to_parquet(str(DOC_METADATA_PATH), index=False, compression="zstd")
meta_size_mb = DOC_METADATA_PATH.stat().st_size / (1024 * 1024)
print(f"✓ Đã ghi Metadata Parquet thành công! Dung lượng: {meta_size_mb:.2f} MB")

### 7. Kiểm thử Tốc độ Truy vấn & Zero-Copy Mmap

In [ ]:
print("⚡ Đang nạp Index qua Zero-Copy MMAP (IO_FLAG_MMAP)...")
t_load = time.time()
# faiss.IO_FLAG_MMAP giúp nạp file 878 MB trong 0.05 giây mà tốn 0 MB RAM!
test_index = faiss.read_index(str(DOC_INDEX_SQ8_PATH), faiss.IO_FLAG_MMAP)
print(f"✓ Nạp index hoàn tất trong {time.time() - t_load:.3f} giây!")

# Tạo 1 query vector ngẫu nhiên để test tốc độ tìm kiếm
fake_query = np.random.randn(1, DIM).astype(np.float32)
faiss.normalize_L2(fake_query)

t_search = time.time()
scores, indices = test_index.search(fake_query, 300)
search_time = time.time() - t_search

print(f"✓ Tìm kiếm Top 300 Documents hoàn tất trong: {search_time * 1000:.2f} ms!")
print(f"Top 5 matched document IDs: {[doc_ids[idx] for idx in indices[0][:5]]}")
print(f"Top 5 document titles: {[titles[idx] for idx in indices[0][:5]]}")

### 8. (Tùy chọn) Tải bộ chỉ mục lên Hugging Face Bucket hoặc Dataset

Chạy cell dưới đây nếu bạn muốn lưu bộ chỉ mục này lên Hugging Face Bucket để khi mở bất kỳ notebook Kaggle nào chỉ cần 1 dòng lệnh tải về là xong.

In [ ]:
import subprocess

if HF_TOKEN:
    print("📤 Đang tải các file chỉ mục lên Hugging Face Bucket: 'nieths/ViBioMIR'...")
    # Đảm bảo CLI hf đã được đăng nhập
    subprocess.run(["hf", "auth", "login", "--token", HF_TOKEN], check=False)
    
    try:
        print("1/2. Đang upload doc_index_sq8.faiss...")
        subprocess.run([
            "hf", "buckets", "cp",
            str(DOC_INDEX_SQ8_PATH),
            "hf://buckets/nieths/ViBioMIR/indices/doc_index_sq8.faiss"
        ], check=True)
        
        print("2/2. Đang upload doc_metadata.parquet...")
        subprocess.run([
            "hf", "buckets", "cp",
            str(DOC_METADATA_PATH),
            "hf://buckets/nieths/ViBioMIR/indices/doc_metadata.parquet"
        ], check=True)
        
        print("✅ UPLOAD THÀNH CÔNG 100% LÊN HF BUCKET!")
    except Exception as e:
        print(f"❌ Lỗi khi upload: {e}")
else:
    print("⚠️ Bỏ qua bước upload vì chưa cấu hình HF_TOKEN.")